In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import calendar
import climatePy
import xarray as xr
import os
from shapely.geometry import Point
from shapely.geometry import box

# load climate catalog
catalog = climatePy.data_catalog()

# location
loc = 'SanMiguel'

# list of GCM models
gcm_list = ['BNU-ESM', 'CCSM4', 'CNRM-CM5', 'CSIRO-Mk3-6-0', 'CanESM2','GFDL-ESM2G', 'HadGEM2-CC365', 'IPSL-CM5A-LR', 'MIROC5', 'MIROC-ESM-CHEM', 
            'MRI-CGCM3', 'NorESM1-M', 'inmcm4']

# load Area of Interest for watershed
#path = 'C:/Users/mcburns/OneDrive - DOI/water-balance/Data/RedwoodCreek'
#AOI = gpd.read_file(os.path.join(path,'RedwoodCreek_shapefile/globalwatershed.shp'))
#path = "C:/Users/mcburns/OneDrive - DOI/Redwood Creek/RWC_analysis_code/"
AOI = gpd.read_file("C:/Users/mcburns/OneDrive - DOI/water-balance/Data/"+loc+"/downloaded_shapefile/Layers/GlobalWatershed.shp")

# Area averages
## MACA future

In [3]:
# GET MACA DATA - FUTURE (AREA)

# pull MACA data
gcm_climate_init = xr.Dataset(climatePy.getMACA(AOI, ['pr','tasmax','tasmin','rsds','vpd','vas','uas'], timeRes='day', model=gcm_list, 
                                                scenario=['rcp45','rcp85'], startDate = '2006-01-01', endDate = '2099-12-31', verbose=False))

# process data into more manageable array format with useful metadata
gcm_climate = gcm_climate_init.mean(['x','y']).to_dataframe().reset_index() 
gcm_climate['date'] = pd.to_datetime(gcm_climate['time'].str.split('_').str[1])
#gcm_climate['Model'] = gcm_climate['time'].str.split('_').str[2]
#gcm_climate['RCP'] = gcm_climate['time'].str.split('_').str[4]
gcm_climate['projection'] = gcm_climate['time'].str.split('_').str[2] + "." + gcm_climate['time'].str.split('_').str[4]
gcm_climate = gcm_climate.drop(columns=['time'])

# get variables into the same dataframe with the same indices
length = len(gcm_climate)
precip = gcm_climate[0:int(length/7)].reset_index().drop(columns=['tasmax','tasmin','rsds','vpd','vas','uas','index'])
tasmax = gcm_climate[int(length*2/7):int(length*3/7)].reset_index().drop(columns=['pr','tasmin','rsds','vpd','vas','uas','index'])
tasmin = gcm_climate[int(length*3/7):int(length*4/7)].reset_index().drop(columns=['pr','tasmax','rsds','vpd','vas','uas','index'])
rsds = gcm_climate[int(length/7):int(length*2/7)].reset_index().drop(columns=['tasmin','tasmax','pr','vpd','vas','uas','index'])
vpd = gcm_climate[int(length*6/7):].reset_index().drop(columns=['tasmin','tasmax','pr','rsds','vas','uas','index'])
vas = gcm_climate[int(length*5/7):int(length*6/7)].reset_index().drop(columns=['tasmin','tasmax','pr','rsds','vpd','uas','index'])
uas = gcm_climate[int(length*4/7):int(length*5/7)].reset_index().drop(columns=['tasmin','tasmax','pr','rsds','vpd','vas','index'])
precip = precip.sort_values(by=['date', 'projection'])
tasmax = tasmax.sort_values(by=['date', 'projection'])
tasmin = tasmin.sort_values(by=['date', 'projection'])
rsds = rsds.sort_values(by=['date', 'projection'])
vpd = vpd.sort_values(by=['date', 'projection'])
vas = vas.sort_values(by=['date', 'projection'])
uas = uas.sort_values(by=['date', 'projection'])

gcm_climate_total = precip.drop(columns=['pr'])
gcm_climate_total['pr'] = precip['pr']    # units are in [mm day-1]
gcm_climate_total['tmmn'] = tasmin['tasmin'] - 273.15  # convert from K to C
gcm_climate_total['tmmx'] = tasmax['tasmax'] - 273.15
gcm_climate_total['srad'] = rsds['rsds']
gcm_climate_total['vpd'] = vpd['vpd']
#gcm_climate_total['vas'] = vas['vas']
#gcm_climate_total['uas'] = uas['uas']
gcm_climate_total['vs'] = np.sqrt(vas['vas']**2 + uas['uas']**2)      # calculate wind speed from east and north wind
#gcm_climate_total['temp_avg'] = (gcm_climate_total['temp_min'] + gcm_climate_total['temp_max'])/2 # calculate average temp

# save to csv file
gcm_climate_total['date'] = gcm_climate_total['date'].dt.strftime('%Y-%m-%d')
gcm_climate_total.to_csv("C:\\Users\\mcburns\\OneDrive - DOI\\water-balance\\Data\\"+loc+"\\MACA_"+loc+"_2006_2100_area.csv", 
                         index=False, date_format="%Y-%m-%d")

## MACA historical

In [4]:
# GET MACA DATA - HISTORICAL (AREA)

# pull MACA data
gcm_climate_init = xr.Dataset(climatePy.getMACA(AOI, ['pr','tasmax','tasmin','rsds','vpd','vas','uas'], timeRes='day', model=gcm_list, 
                                                scenario='Hist', startDate = '1960-01-01', endDate = '2005-12-31', verbose=False))

# process data into more manageable array format with useful metadata
gcm_climate = gcm_climate_init.mean(['x','y']).to_dataframe().reset_index() 
gcm_climate['date'] = pd.to_datetime(gcm_climate['time'].str.split('_').str[1])
#gcm_climate['Model'] = gcm_climate['time'].str.split('_').str[2]
#gcm_climate['RCP'] = gcm_climate['time'].str.split('_').str[4]
gcm_climate['projection'] = gcm_climate['time'].str.split('_').str[2]
gcm_climate = gcm_climate.drop(columns=['time'])

# get variables into the same dataframe with the same indices
length = len(gcm_climate)
precip = gcm_climate[0:int(length/7)].reset_index().drop(columns=['tasmax','tasmin','rsds','vpd','vas','uas','index'])
tasmax = gcm_climate[int(length*2/7):int(length*3/7)].reset_index().drop(columns=['pr','tasmin','rsds','vpd','vas','uas','index'])
tasmin = gcm_climate[int(length*3/7):int(length*4/7)].reset_index().drop(columns=['pr','tasmax','rsds','vpd','vas','uas','index'])
rsds = gcm_climate[int(length/7):int(length*2/7)].reset_index().drop(columns=['tasmin','tasmax','pr','vpd','vas','uas','index'])
vpd = gcm_climate[int(length*6/7):].reset_index().drop(columns=['tasmin','tasmax','pr','rsds','vas','uas','index'])
vas = gcm_climate[int(length*5/7):int(length*6/7)].reset_index().drop(columns=['tasmin','tasmax','pr','rsds','vpd','uas','index'])
uas = gcm_climate[int(length*4/7):int(length*5/7)].reset_index().drop(columns=['tasmin','tasmax','pr','rsds','vpd','vas','index'])
precip = precip.sort_values(by=['date', 'projection'])
tasmax = tasmax.sort_values(by=['date', 'projection'])
tasmin = tasmin.sort_values(by=['date', 'projection'])
rsds = rsds.sort_values(by=['date', 'projection'])
vpd = vpd.sort_values(by=['date', 'projection'])
vas = vas.sort_values(by=['date', 'projection'])
uas = uas.sort_values(by=['date', 'projection'])

gcm_climate_total = precip.drop(columns=['pr'])
gcm_climate_total['pr'] = precip['pr']    # units are in [mm day-1]
gcm_climate_total['tmmn'] = tasmin['tasmin'] - 273.15  # convert from K to C
gcm_climate_total['tmmx'] = tasmax['tasmax'] - 273.15
gcm_climate_total['srad'] = rsds['rsds']
gcm_climate_total['vpd'] = vpd['vpd']
#gcm_climate_total['vas'] = vas['vas']
#gcm_climate_total['uas'] = uas['uas']
gcm_climate_total['vs'] = np.sqrt(vas['vas']**2 + uas['uas']**2)      # calculate wind speed from east and north wind
#gcm_climate_total['temp_avg'] = (gcm_climate_total['temp_min'] + gcm_climate_total['temp_max'])/2 # calculate average temp

# save to csv file
gcm_climate_total['date'] = gcm_climate_total['date'].dt.strftime('%Y-%m-%d')
gcm_climate_total.to_csv("C:\\Users\\mcburns\\OneDrive - DOI\\water-balance\\Data\\"+loc+"\\MACA_"+loc+"_1960_2005_area.csv", 
                         index=False, date_format="%Y-%m-%d")

## GridMET or Daymet historical

In [5]:
# GET GRIDMET/DAYMET - HISTORICAL (AREA)

# getGridMET: available from 1979-01-01 to yesterday, variables are ['tmmn','tmmx','pr']
# getDaymet: available from 1980-01-01 to 2023-12-31, variables are ['tmin','tmax','prcp']
historical_climate = xr.Dataset(climatePy.getGridMET(AOI, ['pr','tmmx','tmmn','srad','vpd','vs'], 
                                                     startDate='1980-01-01', endDate='2022-12-31', verbose=False))
historical_climate = historical_climate.mean(['x','y']).to_dataframe().reset_index() 
historical_climate

historical_climate['date'] = pd.to_datetime(historical_climate['time'].str.split('_').str[1])
historical_climate = historical_climate.drop(columns=['time','crs'])

# get precip, tasmax, and tasmin into the same dataframe with the same indices
length = len(historical_climate)
precip = historical_climate[0:int(length/6)].reset_index().drop(columns=['tmmn','tmmx','srad','vs','vpd','index'])
srad = historical_climate[int(length/6):int(length/6)*2].reset_index().drop(columns=['pr','tmmn','tmmx','vs','vpd','index'])
tmmn = historical_climate[int(length/6)*2:int(length/6)*3].reset_index().drop(columns=['pr','tmmx','srad','vs','vpd','index'])
tmmx = historical_climate[int(length/6)*3:int(length/6)*4].reset_index().drop(columns=['pr','tmmn','srad','vs','vpd','index'])
vpd = historical_climate[int(length/6)*4:int(length/6)*5].reset_index().drop(columns=['pr','tmmx','tmmn','srad','vs','index'])
vs = historical_climate[int(length/6)*5:].reset_index().drop(columns=['pr','tmmn','tmmx','srad','vpd','index'])
historical_climate_total = precip.drop(columns=['pr'])
historical_climate_total['pr'] = precip['pr']    # already in mm
historical_climate_total['tmmn'] = tmmn['tmmn'] - 273.15  # convert from K to C
historical_climate_total['tmmx'] = tmmx['tmmx'] - 273.15
historical_climate_total['srad'] = srad['srad']
historical_climate_total['vpd'] = vpd['vpd']
historical_climate_total['vs'] = vs['vs']
historical_climate_total

#save to csv file
historical_climate_total['date'] = historical_climate_total['date'].dt.strftime('%Y-%m-%d')
#historical_climate_total.to_csv("C:\\Users\\mcburns\\OneDrive - DOI\\water-balance\\Data\\"+loc+"\\GridMET_"+loc+"_1980_2023_area_JN.csv", 
#                                index=False, date_format="%Y-%m-%d")

Found 17009 time intervals
Found 17009 time intervals
Found 17009 time intervals
Found 17009 time intervals
Found 17009 time intervals
Found 17009 time intervals


# Point location 
## MACA future

In [ ]:
# GET MACA DATA - FUTURE (POINT) 

# get point location (center of watershed)
point = gpd.GeoDataFrame(geometry=[Point(AOI.to_crs(epsg=4326).geometry.centroid.iloc[0].x, AOI.to_crs(epsg=4326).geometry.centroid.iloc[0].y)], 
                         crs="EPSG:4326")

# get future data, which starts in 2023 
gcm_climate_init = xr.Dataset(climatePy.getMACA(AOI=point, varname=['pr','tasmax','tasmin','rsds','vpd','vas','uas'], startDate = '2006-01-01', 
                                                endDate = '2099-12-31', timeRes='day', model=gcm_list, scenario=['rcp45','rcp85'],  verbose=False))

# process data into more manageable array format with useful metadata
gcm_climate = gcm_climate_init.to_dataframe().reset_index() 
gcm_climate = gcm_climate.drop(columns=['dim_0'], errors='ignore')  # Remove 'dim_0' if present
gcm_climate = gcm_climate.melt(id_vars='date', var_name='variable', value_name='value')
gcm_climate[['var', 'gcm', 'rcp']] = gcm_climate['variable'].str.extract(r'^(\w+)_([\w\-]+)_r1i1p1_(rcp\d+)$')
gcm_climate['projection'] = gcm_climate['gcm'] + '.' + gcm_climate['rcp']
gcm_climate['date'] = pd.to_datetime(gcm_climate['date'])
gcm_climate = gcm_climate.sort_values(['date', 'projection', 'var'])
gcm_climate_total = gcm_climate.pivot_table(index=['date', 'projection'], columns='var', values='value').reset_index()

# fix variables
gcm_climate_total = gcm_climate_total.rename(columns={'rsds': 'srad', 'tasmax': 'tmmx', 'tasmin': 'tmmn', })
gcm_climate_total['tmmn'] = gcm_climate_total['tmmn'] - 273.15  # convert from K to C
gcm_climate_total['tmmx'] = gcm_climate_total['tmmx'] - 273.15
gcm_climate_total['vs'] = np.sqrt(gcm_climate_total['vas']**2 + gcm_climate_total['uas']**2)      # calculate wind speed from east and north wind
gcm_climate_total = gcm_climate_total.drop(['vas', 'uas'], axis=1)

# save to csv file
gcm_climate_total['date'] = gcm_climate_total['date'].dt.strftime('%Y-%m-%d')
gcm_climate_total.to_csv("C:\\Users\\mcburns\\OneDrive - DOI\\water-balance\\Data\\"+loc+"\\MACA_"+loc+"_2006_2100_point.csv", 
                         index=False, date_format="%Y-%m-%d")

## MACA historical

In [ ]:
# GET MACA DATA - HISTORICAL (POINT) 

# get point location (center of watershed)
point = gpd.GeoDataFrame(geometry=[Point(AOI.to_crs(epsg=4326).geometry.centroid.iloc[0].x, AOI.to_crs(epsg=4326).geometry.centroid.iloc[0].y)], 
                         crs="EPSG:4326")

# get future data, which starts in 2023 
gcm_climate_init = xr.Dataset(climatePy.getMACA(AOI=point, varname=['pr','tasmax','tasmin','rsds','vpd','vas','uas'], startDate = '1960-01-01', 
                                                endDate = '2005-12-31', timeRes='day', model=gcm_list, scenario=['rcp45','rcp85'],  verbose=False))

# process data into more manageable array format with useful metadata
gcm_climate = gcm_climate_init.to_dataframe().reset_index() 
gcm_climate = gcm_climate.drop(columns=['dim_0'], errors='ignore')  # Remove 'dim_0' if present
gcm_climate = gcm_climate.melt(id_vars='date', var_name='variable', value_name='value')
gcm_climate[['var', 'projection']] = gcm_climate['variable'].str.extract(r'(\w+)_([\w\-]+)_r1i1p1_')
gcm_climate['date'] = pd.to_datetime(gcm_climate['date'])
gcm_climate = gcm_climate.sort_values(['date', 'projection', 'var'])
gcm_climate_total = gcm_climate.pivot_table(index=['date', 'projection'], columns='var', values='value').reset_index()

# fix variables
gcm_climate_total = gcm_climate_total.rename(columns={'rsds': 'srad', 'tasmax': 'tmmx', 'tasmin': 'tmmn', })
gcm_climate_total['tmmn'] = gcm_climate_total['tmmn'] - 273.15  # convert from K to C
gcm_climate_total['tmmx'] = gcm_climate_total['tmmx'] - 273.15
gcm_climate_total['vs'] = np.sqrt(gcm_climate_total['vas']**2 + gcm_climate_total['uas']**2)      # calculate wind speed from east and north wind
gcm_climate_total = gcm_climate_total.drop(['vas', 'uas'], axis=1)

# save to csv file
gcm_climate_total['date'] = gcm_climate_total['date'].dt.strftime('%Y-%m-%d')
gcm_climate_total
gcm_climate_total.to_csv("C:\\Users\\mcburns\\OneDrive - DOI\\water-balance\\Data\\"+loc+"\\MACA_"+loc+"_1960_2005_point.csv", 
                         index=False, date_format="%Y-%m-%d")